# 14. Analisis Data (Data Understanding)

Notebook ini memeriksa kualitas dan karakteristik dataset **Diabetes Health Indicators (BRFSS 2015)** sesuai bagian Data Understanding pada proposal:

1. Kualitas data: nilai hilang, tipe data, dan rentang nilai.
2. Duplikat dan konflik label.
3. Statistik deskriptif per kelas.
4. Profil usia dan BMI pada tiap kelas (termasuk kelas Diabetes).
5. Korelasi Spearman.
6. Uji chi-square dengan Cramér's V (fitur kategorik) dan Kruskal-Wallis (fitur kontinu).

**Notebook ini hanya membaca data.** Semua tabel (CSV) dan grafik (PNG) disimpan ke folder baru `hasil_eksperimen/14_analisis_data`. Tidak ada file di `data` atau `artifacts` yang diubah, dan `test_final_locked.csv` tidak dibaca.

Pengaturan `DATA_MODE`:
- `"asli"` (bawaan): seluruh dataset asli, untuk menggambarkan populasi data.
- `"train"`: hanya `train_final.csv`, jika Anda ingin analisis tanpa menyentuh baris yang kemudian menjadi data uji.

Analisis di sini bersifat deskriptif dan tidak dipakai untuk menyetel model.

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas. Aman dijalankan berulang.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

## 1. Pengaturan dan Membaca Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_MODE = "asli"   # "asli" atau "train"

PATH_ASLI = "data/diabetes_012_health_indicators_BRFSS2015.csv"
PATH_TRAIN = "data/final_split/train_final.csv"
OUTPUT_DIR = "hasil_eksperimen/14_analisis_data"
TARGET = "Diabetes_012"

os.makedirs(OUTPUT_DIR, exist_ok=True)

path = PATH_ASLI if DATA_MODE == "asli" else PATH_TRAIN
df = pd.read_csv(path)

# Target dipastikan bernilai bulat 0/1/2 sebelum diubah ke integer
assert (df[TARGET] == df[TARGET].round()).all(), "Target bukan bilangan bulat."
df[TARGET] = df[TARGET].astype(int)

features = [c for c in df.columns if c != TARGET]
NAMA_KELAS = {0: "Tidak diabetes", 1: "Prediabetes", 2: "Diabetes"}

print("Sumber data :", path)
print("Bentuk data :", df.shape)
print("Jumlah fitur:", len(features))

dist = df[TARGET].value_counts().sort_index()
tabel_kelas = pd.DataFrame({
    "Kelas": [NAMA_KELAS[int(k)] for k in dist.index],
    "Jumlah": dist.values,
    "Persen": (dist.values / len(df) * 100).round(2)
})
print("\nDistribusi kelas:")
print(tabel_kelas.to_string(index=False))
tabel_kelas.to_csv(os.path.join(OUTPUT_DIR, "distribusi_kelas.csv"), index=False)

## 2. Kualitas Data: Nilai Hilang, Tipe, dan Rentang Nilai

Rentang acuan mengikuti kodebook BRFSS untuk dataset ini (variabel biner 0/1, `BMI` 12-98, `GenHlth` 1-5, `MentHlth` dan `PhysHlth` 0-30 hari, `Age` kategori 1-13, `Education` 1-6, `Income` 1-8). Nilai di luar rentang acuan ditandai, bukan dihapus.

In [ ]:
BINER_01 = (0, 1)
RENTANG = {
    TARGET: (0, 2),
    "HighBP": BINER_01, "HighChol": BINER_01, "CholCheck": BINER_01,
    "BMI": (12, 98),
    "Smoker": BINER_01, "Stroke": BINER_01, "HeartDiseaseorAttack": BINER_01,
    "PhysActivity": BINER_01, "Fruits": BINER_01, "Veggies": BINER_01,
    "HvyAlcoholConsump": BINER_01, "AnyHealthcare": BINER_01, "NoDocbcCost": BINER_01,
    "GenHlth": (1, 5), "MentHlth": (0, 30), "PhysHlth": (0, 30),
    "DiffWalk": BINER_01, "Sex": BINER_01,
    "Age": (1, 13), "Education": (1, 6), "Income": (1, 8),
}

baris = []
for c in df.columns:
    s = df[c]
    ada_acuan = c in RENTANG
    lo, hi = RENTANG.get(c, (np.nan, np.nan))
    di_luar = int(((s < lo) | (s > hi)).sum()) if ada_acuan else np.nan
    baris.append({
        "Kolom": c,
        "Tipe": str(s.dtype),
        "Hilang": int(s.isna().sum()),
        "Min": s.min(),
        "Maks": s.max(),
        "Nilai_Unik": int(s.nunique()),
        "Rentang_Acuan": f"{lo}-{hi}" if ada_acuan else "-",
        "Di_Luar_Rentang": di_luar,
        "Bukan_Bulat": int((s != s.round()).sum()),
    })

kualitas = pd.DataFrame(baris)
print(kualitas.to_string(index=False))
kualitas.to_csv(os.path.join(OUTPUT_DIR, "kualitas_rentang_nilai.csv"), index=False)

print("\nTotal nilai hilang           :", int(df.isna().sum().sum()))
print("Total nilai di luar rentang  :", int(kualitas["Di_Luar_Rentang"].fillna(0).sum()))
print("Kolom tanpa rentang acuan    :", [c for c in df.columns if c not in RENTANG])

## 3. Duplikat dan Konflik Label

- **Duplikat persis**: seluruh kolom sama, termasuk label.
- **Konflik label**: kombinasi 21 fitur yang sama muncul dengan label berbeda.

BRFSS tidak memiliki ID responden, sehingga dua baris identik belum tentu satu orang yang tercatat dua kali. Bagian ini hanya menghitung. Keputusan menghapus atau mempertahankan duplikat adalah keputusan metodologis.

In [ ]:
dup_extra = df.duplicated()
dup_semua = df.duplicated(keep=False)

n = len(df)
print("Baris kelebihan (salinan)  :", int(dup_extra.sum()), f"({dup_extra.mean()*100:.2f}%)")
print("Baris yang punya kembaran  :", int(dup_semua.sum()))
salinan_kelas = df.loc[dup_extra, TARGET].value_counts().reindex([0, 1, 2], fill_value=0)

g = df.groupby(features)[TARGET].agg(["size", "nunique"]).reset_index()
berulang = g[g["size"] > 1]
konflik = berulang[berulang["nunique"] > 1]

print("\nPola fitur yang muncul >1 kali :", len(berulang))
print("  - label semua sama (duplikat)  :", int((berulang["nunique"] == 1).sum()))
print("  - label berbeda (konflik)      :", len(konflik))
print("Baris yang terlibat konflik      :", int(konflik["size"].sum()),
      f"({konflik['size'].sum()/n*100:.2f}%)")

baris_konflik = df.merge(konflik[features], on=features, how="inner")
konflik_kelas = baris_konflik[TARGET].value_counts().reindex([0, 1, 2], fill_value=0)
jumlah_kelas = df[TARGET].value_counts().reindex([0, 1, 2])

ringkas_dup = pd.DataFrame({
    "Kelas": [NAMA_KELAS[k] for k in [0, 1, 2]],
    "Jumlah_Baris": jumlah_kelas.values,
    "Salinan_Persis": salinan_kelas.values,
    "Salinan_Persen_Kelas": (salinan_kelas.values / jumlah_kelas.values * 100).round(2),
    "Baris_Konflik": konflik_kelas.values,
    "Konflik_Persen_Kelas": (konflik_kelas.values / jumlah_kelas.values * 100).round(2),
})
print()
print(ringkas_dup.to_string(index=False))
ringkas_dup.to_csv(os.path.join(OUTPUT_DIR, "duplikat_dan_konflik.csv"), index=False)

## 4. Statistik Deskriptif per Kelas

Fitur dikelompokkan otomatis berdasarkan jumlah nilai uniknya: **biner** (<= 2 nilai), **ordinal** (3-15 nilai), dan **kontinu** (> 15 nilai).

In [ ]:
nun = df[features].nunique()
biner = nun[nun <= 2].index.tolist()
ordinal = nun[(nun > 2) & (nun <= 15)].index.tolist()
kontinu = nun[nun > 15].index.tolist()

print("Biner    :", biner)
print("Ordinal  :", ordinal)
print("Kontinu  :", kontinu)


def ringkas_numerik(cols):
    hasil = []
    for c in cols:
        baris = {"Fitur": c}
        for k in [0, 1, 2]:
            s = df.loc[df[TARGET] == k, c]
            baris[f"Mean_K{k}"] = s.mean()
            baris[f"Median_K{k}"] = s.median()
            baris[f"SD_K{k}"] = s.std()
        s = df[c]
        baris["Mean_Semua"] = s.mean()
        baris["SD_Semua"] = s.std()
        baris["Min"] = s.min()
        baris["Maks"] = s.max()
        hasil.append(baris)
    return pd.DataFrame(hasil)


stat_num = ringkas_numerik(kontinu + ordinal)
print("\nStatistik fitur kontinu dan ordinal (K0 = tidak diabetes, K1 = prediabetes, K2 = diabetes):")
print(stat_num.round(2).to_string(index=False))
stat_num.round(4).to_csv(os.path.join(OUTPUT_DIR, "statistik_numerik_per_kelas.csv"), index=False)

prev = (df.groupby(TARGET)[biner].mean().T * 100)
prev.columns = [NAMA_KELAS[int(c)] for c in prev.columns]
prev["Semua"] = df[biner].mean() * 100
print("\nPersentase jawaban '1' pada fitur biner, per kelas:")
print(prev.round(2).to_string())
prev.round(3).to_csv(os.path.join(OUTPUT_DIR, "prevalensi_fitur_biner_per_kelas.csv"))

## 5. Profil Usia dan BMI

`Age` adalah kategori usia BRFSS (1 = 18-24 tahun, 2 = 25-29, ... 13 = 80 tahun ke atas). Tabel pertama menjawab "seperti apa sebaran usia pada tiap kelas", tabel kedua menjawab "berapa persen tiap kelas pada setiap kelompok usia".

In [ ]:
LABEL_USIA = {1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39", 5: "40-44", 6: "45-49", 7: "50-54",
              8: "55-59", 9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80+"}

usia = df["Age"].astype(int)
usia_dalam_kelas = pd.crosstab(usia, df[TARGET], normalize="columns") * 100
usia_dalam_kelas.columns = [NAMA_KELAS[int(c)] for c in usia_dalam_kelas.columns]
usia_dalam_kelas.index = [f"{i} ({LABEL_USIA.get(i, '?')})" for i in usia_dalam_kelas.index]

prevalensi_usia = pd.crosstab(usia, df[TARGET], normalize="index") * 100
prevalensi_usia.columns = [NAMA_KELAS[int(c)] for c in prevalensi_usia.columns]
prevalensi_usia.index = [f"{i} ({LABEL_USIA.get(i, '?')})" for i in prevalensi_usia.index]

print("Sebaran usia di dalam tiap kelas (% per kelas):")
print(usia_dalam_kelas.round(2).to_string())
print("\nKomposisi kelas pada tiap kelompok usia (% per kelompok usia):")
print(prevalensi_usia.round(2).to_string())
usia_dalam_kelas.round(3).to_csv(os.path.join(OUTPUT_DIR, "usia_sebaran_dalam_kelas.csv"))
prevalensi_usia.round(3).to_csv(os.path.join(OUTPUT_DIR, "usia_komposisi_kelas.csv"))

bins = [0, 18.5, 25, 30, 35, 40, np.inf]
label_bmi = ["<18,5 (kurus)", "18,5-24,9 (normal)", "25-29,9 (berat lebih)",
             "30-34,9 (obesitas I)", "35-39,9 (obesitas II)", ">=40 (obesitas III)"]
bmi_kat = pd.cut(df["BMI"], bins=bins, labels=label_bmi, right=False)

bmi_dalam_kelas = pd.crosstab(bmi_kat, df[TARGET], normalize="columns") * 100
bmi_dalam_kelas.columns = [NAMA_KELAS[int(c)] for c in bmi_dalam_kelas.columns]
prevalensi_bmi = pd.crosstab(bmi_kat, df[TARGET], normalize="index") * 100
prevalensi_bmi.columns = [NAMA_KELAS[int(c)] for c in prevalensi_bmi.columns]

print("\nSebaran kategori BMI di dalam tiap kelas (% per kelas):")
print(bmi_dalam_kelas.round(2).to_string())
print("\nKomposisi kelas pada tiap kategori BMI (% per kategori):")
print(prevalensi_bmi.round(2).to_string())
bmi_dalam_kelas.round(3).to_csv(os.path.join(OUTPUT_DIR, "bmi_sebaran_dalam_kelas.csv"))
prevalensi_bmi.round(3).to_csv(os.path.join(OUTPUT_DIR, "bmi_komposisi_kelas.csv"))

ringkas_bmi = df.groupby(TARGET)["BMI"].describe().round(2)
ringkas_bmi.index = [NAMA_KELAS[int(i)] for i in ringkas_bmi.index]
print("\nRingkasan BMI per kelas:")
print(ringkas_bmi.to_string())
ringkas_bmi.to_csv(os.path.join(OUTPUT_DIR, "bmi_ringkasan_per_kelas.csv"))

WARNA = {0: "#4C78A8", 1: "#F58518", 2: "#B6362E"}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))

for k in [0, 1, 2]:
    ax[0].plot(range(1, 14), usia_dalam_kelas.iloc[:, k].values, marker="o", color=WARNA[k], label=NAMA_KELAS[k])
ax[0].set_title("Sebaran kategori usia di dalam tiap kelas")
ax[0].set_xlabel("Kategori usia BRFSS (1 = 18-24, 13 = 80+)")
ax[0].set_ylabel("% dari kelas")
ax[0].legend()

for k in [0, 1, 2]:
    s = df.loc[df[TARGET] == k, "BMI"]
    ax[1].hist(s, bins=np.arange(10, 70, 1), density=True, alpha=0.45, color=WARNA[k], label=NAMA_KELAS[k])
ax[1].set_title("Sebaran BMI per kelas (densitas)")
ax[1].set_xlabel("BMI")
ax[1].legend()

lebar = 0.38
pos = np.arange(len(label_bmi))
ax[2].bar(pos - lebar / 2, prevalensi_bmi["Prediabetes"].values, width=lebar, color=WARNA[1], label="Prediabetes")
ax[2].bar(pos + lebar / 2, prevalensi_bmi["Diabetes"].values, width=lebar, color=WARNA[2], label="Diabetes")
ax[2].set_xticks(pos)
ax[2].set_xticklabels([l.split(" ")[0] for l in label_bmi], rotation=30)
ax[2].set_title("% Prediabetes dan Diabetes per kategori BMI")
ax[2].set_ylabel("% dalam kategori BMI")
ax[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "profil_usia_bmi.png"), dpi=150)
plt.show()

## 6. Korelasi Spearman

Korelasi Spearman dihitung antar fitur dan antara fitur dengan target. Target diperlakukan sebagai ordinal (0 < 1 < 2). Anggapan ini hanya pendekatan: Prediabetes belum tentu berada "di tengah" secara linear pada setiap fitur, jadi korelasi dengan target jangan dibaca sebagai efek yang tegas.

In [ ]:
kolom_korelasi = features + [TARGET]
corr = df[kolom_korelasi].corr(method="spearman")
corr.round(4).to_csv(os.path.join(OUTPUT_DIR, "korelasi_spearman.csv"))

corr_target = corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
print("Korelasi Spearman fitur terhadap target (diurutkan menurut nilai mutlak):")
print(corr_target.round(4).to_string())
corr_target.round(4).to_csv(os.path.join(OUTPUT_DIR, "korelasi_spearman_dengan_target.csv"), header=["Spearman"])

pasangan = []
for i, a in enumerate(features):
    for b in features[i + 1:]:
        r = corr.loc[a, b]
        if abs(r) >= 0.4:
            pasangan.append((a, b, round(float(r), 4)))
pasangan.sort(key=lambda t: -abs(t[2]))
print("\nPasangan fitur dengan |Spearman| >= 0.4 (indikasi keterkaitan antar fitur):")
for a, b, r in pasangan:
    print(f"  {a:22s} - {b:22s} : {r:+.4f}")
if not pasangan:
    print("  (tidak ada)")
pd.DataFrame(pasangan, columns=["Fitur_1", "Fitur_2", "Spearman"]).to_csv(
    os.path.join(OUTPUT_DIR, "pasangan_fitur_berkorelasi.csv"), index=False)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(kolom_korelasi)))
ax.set_yticks(range(len(kolom_korelasi)))
ax.set_xticklabels(kolom_korelasi, rotation=90)
ax.set_yticklabels(kolom_korelasi)
for i in range(len(kolom_korelasi)):
    for j in range(len(kolom_korelasi)):
        v = corr.values[i, j]
        if abs(v) >= 0.3 and i != j:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if abs(v) > 0.6 else "black")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Korelasi Spearman")
ax.set_title("Korelasi Spearman antar fitur dan target")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "heatmap_spearman.png"), dpi=150)
plt.show()

## 7. Chi-square dan Cramér's V (fitur biner dan ordinal), Kruskal-Wallis (fitur kontinu)

Pada jumlah data sebesar ini, nilai p hampir selalu sangat kecil sehingga tidak informatif. **Gunakan ukuran efek**: Cramér's V (ambang Cohen menyesuaikan derajat kebebasan) dan epsilon-kuadrat untuk Kruskal-Wallis.

In [ ]:
def ambang_cramer(dfs):
    # Ambang Cohen untuk Cramér's V: (kecil, sedang, besar) menurut min(baris, kolom) - 1
    return {1: (0.1, 0.3, 0.5), 2: (0.07, 0.21, 0.35)}.get(dfs, (0.06, 0.17, 0.29))


def label_efek(nilai, ambang):
    kecil, sedang, besar = ambang
    if nilai < kecil:
        return "diabaikan"
    if nilai < sedang:
        return "kecil"
    if nilai < besar:
        return "sedang"
    return "besar"


hasil_chi = []
for c in biner + ordinal:
    ct = pd.crosstab(df[c], df[TARGET])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    total = ct.values.sum()
    dfs = min(ct.shape) - 1
    v = float(np.sqrt(chi2 / (total * dfs)))
    hasil_chi.append({
        "Fitur": c,
        "Tipe": "biner" if c in biner else "ordinal",
        "Chi2": chi2,
        "DF": dof,
        "p_value": p,
        "Cramers_V": v,
        "Ukuran_Efek": label_efek(v, ambang_cramer(dfs)),
    })

tabel_chi = pd.DataFrame(hasil_chi).sort_values("Cramers_V", ascending=False).reset_index(drop=True)
print("Chi-square dan Cramér's V terhadap target (3 kelas):")
print(tabel_chi.assign(Chi2=tabel_chi["Chi2"].round(1), Cramers_V=tabel_chi["Cramers_V"].round(4),
                       p_value=tabel_chi["p_value"].map(lambda x: f"{x:.2e}")).to_string(index=False))
tabel_chi.to_csv(os.path.join(OUTPUT_DIR, "chi_square_cramers_v.csv"), index=False)

hasil_kw = []
for c in kontinu:
    kelompok = [df.loc[df[TARGET] == k, c].to_numpy() for k in [0, 1, 2]]
    H, p = stats.kruskal(*kelompok)
    eps2 = float(H / (len(df) - 1))
    hasil_kw.append({
        "Fitur": c,
        "H": H,
        "p_value": p,
        "Epsilon2": eps2,
        "Ukuran_Efek": label_efek(eps2, (0.01, 0.08, 0.26)),
    })

tabel_kw = pd.DataFrame(hasil_kw).sort_values("Epsilon2", ascending=False).reset_index(drop=True)
print("\nKruskal-Wallis fitur kontinu terhadap target:")
if len(tabel_kw):
    print(tabel_kw.assign(H=tabel_kw["H"].round(1), Epsilon2=tabel_kw["Epsilon2"].round(4),
                          p_value=tabel_kw["p_value"].map(lambda x: f"{x:.2e}")).to_string(index=False))
tabel_kw.to_csv(os.path.join(OUTPUT_DIR, "kruskal_wallis_kontinu.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 7))
t = tabel_chi.sort_values("Cramers_V")
ax.barh(t["Fitur"], t["Cramers_V"], color="#4C78A8")
ax.set_xlabel("Cramér's V terhadap target")
ax.set_title("Ukuran efek fitur biner dan ordinal terhadap kelas diabetes")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cramers_v.png"), dpi=150)
plt.show()

## 8. Ringkasan Otomatis

Sel ini mencetak ringkasan angka kunci dan menyimpannya ke `ringkasan_analisis_data.txt`. Ringkasan ini sekadar bantuan menulis laporan, bukan kesimpulan.

In [ ]:
baris_ringkas = []
baris_ringkas.append(f"Sumber data: {path} ({len(df):,} baris, {len(features)} fitur)")
baris_ringkas.append("Distribusi kelas: " + "; ".join(
    f"{r.Kelas} {r.Jumlah:,} ({r.Persen}%)" for r in tabel_kelas.itertuples()))
baris_ringkas.append(f"Nilai hilang: {int(df.isna().sum().sum())}")
baris_ringkas.append(f"Nilai di luar rentang acuan: {int(kualitas['Di_Luar_Rentang'].fillna(0).sum())}")
baris_ringkas.append(f"Duplikat persis (salinan): {int(dup_extra.sum()):,} ({dup_extra.mean()*100:.2f}%); "
                     f"per kelas 0/1/2 = {salinan_kelas.tolist()}")
baris_ringkas.append(f"Pola fitur dengan label berbeda (konflik): {len(konflik):,}; "
                     f"baris terlibat {int(konflik['size'].sum()):,}; per kelas 0/1/2 = {konflik_kelas.tolist()}")
baris_ringkas.append("Fitur dengan |Spearman| terbesar terhadap target: " + ", ".join(
    f"{k} ({v:+.3f})" for k, v in corr_target.head(5).items()))
baris_ringkas.append("Fitur dengan Cramér's V terbesar: " + ", ".join(
    f"{r.Fitur} ({r.Cramers_V:.3f})" for r in tabel_chi.head(5).itertuples()))
if len(tabel_kw):
    baris_ringkas.append("Kruskal-Wallis (epsilon2): " + ", ".join(
        f"{r.Fitur} ({r.Epsilon2:.3f})" for r in tabel_kw.itertuples()))

teks = "\n".join(baris_ringkas)
print(teks)
with open(os.path.join(OUTPUT_DIR, "ringkasan_analisis_data.txt"), "w", encoding="utf-8") as f:
    f.write(teks + "\n")

print("\nSemua hasil disimpan di:", OUTPUT_DIR)
print("CATATAN: notebook ini tidak membaca test_final_locked.csv dan tidak menulis ke data atau artifacts.")

## 9. Cara Memakai Hasil untuk Laporan

- **Kualitas data**: laporkan jumlah nilai hilang, nilai di luar rentang, jumlah duplikat persis, dan konflik label. Jelaskan keputusan terhadap duplikat beserta alasannya.
- **Statistik per kelas dan profil usia/BMI**: gunakan tabel `statistik_numerik_per_kelas.csv`, `usia_*`, dan `bmi_*`.
- **Spearman**: pasangan fitur yang saling berkorelasi tinggi bisa disebut sebagai catatan keterkaitan antar fitur. Model pohon tidak terlalu terpengaruh oleh hal ini.
- **Cramér's V dan epsilon-kuadrat**: tafsirkan dari ukuran efek, bukan dari nilai p. Ukuran efek kecil pada semua fitur adalah temuan yang wajar untuk data survei seperti ini dan konsisten dengan sulitnya memisahkan Prediabetes.
- Hubungan yang ditemukan bersifat asosiasi, bukan sebab-akibat.